In [ ]:
#1. Pra-pemrosesan Data (Pembersihan dan Penggabungan Data)
import pandas as pd
data_eur=pd.read_csv('eur_usd.csv',sep=None)
data_gbp=pd.read_csv('gbp_usd.csv',sep=None)
data_eur['datetime']=pd.to_datetime(data_eur['date']+' '+data_eur['time'])
data_eur.set_index('datetime',inplace=True)
data_gbp['datetime']=pd.to_datetime(data_gbp['date']+' '+data_gbp['time'])
data_gbp.set_index('datetime',inplace=True)
#data_eur.to_csv('eur_usd2.csv',index=True)
#data_gbp.to_csv('gbp_usd2.csv',index=True)

In [ ]:
#2. Analisis Data (Menentukan Rasio Persentase (%) Latih : Uji, yang terbaik berdasarkan nilai MAPE (%) terkecil serta validasi dengan Time-series Cross-validation)
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
import warnings
warnings.filterwarnings("ignore")


# Konfigurasi Dataset
EUR_PATH='eur_usd2.csv'
GBP_PATH='gbp_usd2.csv'
RESAMPLE_RULE='1H'


#Persentase data latih
TRAIN_SIZE=0.97


SEASONAL=False
def mape(y_true,y_pred):
    y_true,y_pred=np.array(y_true),np.array(y_pred)
    mask=y_true!=0
    if mask.sum()==0:
        return np.nan
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100
def time_series_cv(series,order,n_splits=5):
    n=len(series)
    fold_size=n // (n_splits+1)
    mape_scores=[]
    for i in range(1,n_splits+1):
        split_point=n-(n_splits-i+1)*fold_size
        if split_point<=0 or split_point>=n:
            continue
        train_cv=series.iloc[:split_point]
        test_cv=series.iloc[split_point:split_point+fold_size]
        if len(test_cv)==0:
            continue
        try:
            model_cv=ARIMA(train_cv,order=order).fit()
            pred_cv=model_cv.get_forecast(steps=len(test_cv)).predicted_mean
            mape_cv=mape(test_cv.values,pred_cv.values)
            mape_scores.append(mape_cv)
        except:
            continue
    return np.mean(mape_scores) if mape_scores else np.nan
def run_fast_arima(csv_path,name,train_size=TRAIN_SIZE,resample_rule=RESAMPLE_RULE,seasonal=SEASONAL):
    df=pd.read_csv(csv_path,parse_dates=['datetime'])
    df.set_index('datetime',inplace=True)
    series=df['close'].astype(float).sort_index().resample(resample_rule).mean().dropna()
    n=len(series)
    split=int(n*train_size)
    train,test=series.iloc[:split],series.iloc[split:]
    order=pm.auto_arima(
        train, 
        start_p=0,start_q=0,max_p=5,max_q=5,
        seasonal=seasonal,m=1,stepwise=True,trace=False,error_action='ignore'
    ).order
    fitted=ARIMA(train,order=order).fit()
    pred_mean=fitted.get_forecast(steps=len(test)).predicted_mean
    mape_result=mape(test.values,pred_mean.values)
    

    # Time-series Cross-validation
    cv_score=time_series_cv(series,order,n_splits=24) #n_splits=24: TSCV menguji 24-fold bagian data


    print(f"Dataset: {len(series)} titik data | Latih: {len(train)} ({train_size*100:.1f}%) | Uji: {len(test)} ({(1-train_size)*100:.1f}%)")
    print(f"Urutan ARIMA terbaik: {order}")
    print(f"MAPE pada data latih-uji (%): {mape_result:.4f}")
    print(f"MAPE dari Time-series Cross-validation 24-fold (%): {cv_score:.4f}\n")
    return mape_result
print(f"Analisis ARIMA dengan Rasio Latih : Uji {TRAIN_SIZE*100:.0f}%:{(1-TRAIN_SIZE)*100:.0f}%")
print("-"*50)
print(f"Interval Waktu Resample: {RESAMPLE_RULE}")
print("EUR / USD:")
mape_eur=run_fast_arima(EUR_PATH,name='EUR-USD')
print("GBP / USD:")
mape_gbp=run_fast_arima(GBP_PATH,name='GBP-USD')

In [ ]:
#3a. Pe-modelan Auto-regressive Integrated Moving Average (ARIMA)
import warnings
warnings.filterwarnings("ignore")
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
from sklearn.metrics import mean_absolute_error,mean_squared_error
plt.rcParams['figure.figsize']=(14,5)
plt.rcParams['axes.grid']=True


#3b. Fungsi Utilitas Umum
def rmse(y_true,y_pred):
    return np.sqrt(mean_squared_error(y_true,y_pred))
def mape(y_true,y_pred):
    y_true,y_pred=np.array(y_true),np.array(y_pred)
    mask=y_true!=0
    if mask.sum()==0:
        return np.nan
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100
def evaluate_forecast(y_true,y_pred):
    return{
        'MAPE(%)': mape(y_true,y_pred)
    }


#3c. Pra-pemrosesan Data
def load_and_index(csv_path,datetime_col='datetime',close_col='close'):
    df=pd.read_csv(csv_path,parse_dates=[datetime_col])
    df.set_index(datetime_col,inplace=True)
    df.sort_index(inplace=True)
    series=df[close_col].astype(float)
    return series,df
def resample_series(series,rule='1H',how='mean'):
    if how=='ohlc':
        return series.resample(rule).ohlc()
    else:
        return series.resample(rule).mean()
def time_train_test_split(series,train_size=0.99):
    n=len(series)
    split=int(n*train_size)
    train=series.iloc[:split]
    test=series.iloc[split:]
    return train,test


#3d. Uji Stasioneritas & Differencing
def adf_test(series,signif=0.05,verbose=False):
    series=series.dropna()
    result=adfuller(series,autolag='AIC')
    pvalue=result[1]
    is_stat=pvalue<signif
    if verbose:
        print(f"Statistik ADF: {result[0]:.4f}, Nilai-p: {pvalue:.4f}, Stasioner: {is_stat}")
    return is_stat,pvalue
def difference_until_stationary(series,max_d=3,signif=0.05,verbose=False):
    s=series.copy()
    d=0
    is_stat,p=adf_test(s,signif=signif,verbose=verbose)
    while not is_stat and d<max_d:
        s=s.diff().dropna()
        d+=1
        is_stat,p=adf_test(s,signif=signif,verbose=verbose)
    return d,s


#3e. Pemodelan ARIMA
def auto_arima_select(train_series,seasonal=False,m=1,max_p=5,max_q=5):
    model=pm.auto_arima(
        train_series,
        start_p=0,start_q=0,
        max_p=max_p,max_q=max_q,
        seasonal=seasonal,m=m,
        stepwise=True,
        trace=False,
        error_action='ignore',
        suppress_warnings=True
    )
    return model
def fit_arima_model(train_series,order):
    model=ARIMA(train_series,order=order)
    fitted=model.fit()
    return fitted
def forecast_arima(fitted_model,steps):
    forecast_res=fitted_model.get_forecast(steps=steps)
    mean=forecast_res.predicted_mean
    ci=forecast_res.conf_int()
    return mean,ci


#3f. Visualisasi
def plot_series_and_forecast(series,train,test,pred,ci=None,title=''):
    plt.figure(figsize=(14,5))
    plt.plot(series.index,series.values,label='Aktual',color='gray',alpha=0.4)
    plt.plot(train.index,train.values,label='Latih (%)',color='tab:blue')
    plt.plot(test.index,test.values,label='Uji (%)',color='tab:orange')
    plt.plot(pred.index,pred.values,label='Prediksi',color='tab:green')
    if ci is not None:
        plt.fill_between(ci.index,ci.iloc[:,0],ci.iloc[:,1],color='green',alpha=0.2)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


#3g. Pipeline Lengkap untuk 1 Dataset
def run_pipeline(csv_path,name,resample_rule='1H',train_size=0.99,seasonal=False,m=24):
    print(f"\n=== Memproses {name} ===")
    series,df=load_and_index(csv_path)
    if resample_rule:
        series=resample_series(series,rule=resample_rule)
    series=series.dropna()
    train,test=time_train_test_split(series,train_size)
    print(f"Titik data: {len(series)} | Latih: {len(train)} | Uji: {len(test)}")
    d_suggested,diffed=difference_until_stationary(train,verbose=False)
    print(f"differencing (d) yang disarankan: {d_suggested}")
    auto_model=auto_arima_select(train,seasonal=seasonal,m=m)
    order=auto_model.order
    print(f"Titik data: {len(series)} | Latih: {len(train)} | Uji: {len(test)}")
    print(f"Urutan ARIMA terbaik: {order}")
    fitted=fit_arima_model(train,order)
    steps=len(test)
    pred_mean,pred_ci=forecast_arima(fitted,steps)
    pred_mean.index=test.index
    pred_ci.index=test.index
    metrics=evaluate_forecast(test.values,pred_mean.values)
    print("Evaluasi:",metrics)
    plot_series_and_forecast(series,train,test,pred_mean,ci=pred_ci,title=f"{name} ARIMA{order}")
    return {
        'name': name,
        'order': order,
        'metrics': metrics,
        'model': fitted,
        'predictions': pred_mean
    }


#3h.Jalankan untuk Kedua Dataset
eur_path='eur_usd2.csv'
gbp_path='gbp_usd2.csv'
resample_rule='1H'   # ubah ke '1D' jika ingin harian
seasonal=False       # ubah ke True jika ada pola musiman (m=24 untuk hourly)
m=24
res_eur=run_pipeline(eur_path,name='EUR-USD',resample_rule=resample_rule,seasonal=seasonal,m=m)
res_gbp=run_pipeline(gbp_path,name='GBP-USD',resample_rule=resample_rule,seasonal=seasonal,m=m)


#3j. Simpan Model & Hasil Prediksi
os.makedirs('prediksi',exist_ok=True)
joblib.dump(res_eur['model'],'prediksi/model_eur_usd.joblib')
joblib.dump(res_gbp['model'],'prediksi/model_gbp_usd.joblib')
res_eur['predictions'].to_csv('prediksi/prep_eur_usd.csv')
res_gbp['predictions'].to_csv('prediksi/prep_gbp_usd.csv')
print("Model dan hasil prediksi disimpan di folder 'prediksi'")

In [ ]:
#4. Implementasi prediksi menggunakan Model ARIMA yang telah disimpan (Rolling / Walk-forward Forecast) untuk prediksi yang lebih realistis
import warnings
warnings.filterwarnings("ignore")
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
plt.rcParams['figure.figsize']=(12,5)


#Konfigurasi CSV
model_path=Path('prediksi/model_gbp_usd.joblib')
csv_path=Path('eur_usd2.csv')


model=joblib.load(model_path)


#Urutan ARIMA (Mis. 1,2,3)
order=(0,1,1)
print("Urutan ARIMA Terbaik:",order)


df=pd.read_csv(csv_path,parse_dates=['datetime'])
df.set_index('datetime',inplace=True)
series=df['close'].astype(float).sort_index()
print("Panjang Series:",len(series))


#Parameter Prediksi
resample_rule='1H'   #Prediksi tiap jam
K=24                 #Re-fit setiap 24 langkah
H_total=72          #Jumlah prediksi 72 Jam (3 Hari)


history=series.copy()
pred_values=[]
pred_idx=[]
steps_done=0
while steps_done<H_total:
    try:
        fitted=ARIMA(history,order=order).fit()
    except Exception as e:
        print("Error saat fitting ARIMA:",e)
        break
    step_size=min(K,H_total-steps_done)
    fut=fitted.get_forecast(steps=step_size).predicted_mean
    last_ts=history.index[-1]


    #Bangun datetime index
    if resample_rule=='1H':
        idx_chunk=pd.date_range(start=last_ts+pd.Timedelta(hours=1),
                                  periods=step_size,freq='H')
    elif resample_rule=='1D':
        idx_chunk=pd.date_range(start=last_ts+pd.Timedelta(days=1),
                                  periods=step_size,freq='D')
    else:
        idx_chunk=range(len(pred_idx),len(pred_idx)+step_size)
    fut.index=idx_chunk


    pred_values.extend(fut.values)
    pred_idx.extend(list(idx_chunk))
    history=pd.concat([history,pd.Series(fut.values,index=fut.index)])
    steps_done+=step_size
    print(f"Proses: {steps_done}/{H_total}")


#Simpan hasil prediksi ke CSV
pred_series=pd.Series(pred_values,index=pred_idx)
output_file=Path('prediksi/prediksi_eur_usd.csv')
pred_series.to_csv(output_file)
print("Disimpan ke:", output_file)


#Plot
plt.figure()
plt.plot(series[-200:],label="Aktual (200-an ke belakang)")
plt.plot(pred_series, label="Prediksi Rolling / Walk-forward")
plt.title("ARIMA (Prediksi Rolling / Walk-forward)")
plt.legend()
plt.show()

In [ ]:
#5 (Opsional). Implementasi pembaruan Model ARIMA yang telah disimpan (Re-train)
import warnings
warnings.filterwarnings("ignore")
import joblib
import pandas as pd
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA


#Konfigurasi CSV
model_path=Path('prediksi/model_eur_usd.joblib')
csv_path_old=Path('eur_usd2.csv') #Dataset lama
csv_path_new=Path('update_eur_usd.csv')  #Dataset terbaru


model=joblib.load(model_path)
df_old=pd.read_csv(csv_path_old,parse_dates=['datetime'])
df_old.set_index('datetime',inplace=True)
series_old=df_old['close'].astype(float).sort_index()
print("Panjang Series Dataset Lama:",len(series_old))
df_new=pd.read_csv(csv_path_new,parse_dates=['datetime'])
df_new.set_index('datetime',inplace=True)
series_new=df_new['close'].astype(float).sort_index()
print("Panjang Series Dataset Baru:",len(series_new))
combined = pd.concat([series_old, series_new]).sort_index()
print("Panjang Series Gabungan:",len(combined))


#Urutan ARIMA (Mis. 1,2,3)
order=(0,1,2)
print("Urutan ARIMA Terbaik:",order)


print("Fitting ARIMA...")
fitted_new=ARIMA(combined,order=order).fit()


#Simpan hasil ke CSV
output_path=Path('prediksi/update_model_eur_usd.joblib')
joblib.dump(fitted_new,output_path)
print("Re-train Model berhasil dan disimpan ke:",output_path)